# GC-SWGAN — Member 3 final held-out test evaluation

**This notebook does NOT train or tune a model.** It first verifies the SHA-256 dataset and exact corrected split, selected validation-best checkpoints, and training-code commit. It then gives a **separate, explicit one-time test inference step**.

Selected validation-best classifiers (from Member 2): **A step 8,500, validation macro-F1 0.1918**; **B step 10,000, validation macro-F1 0.2797**. These are **not test metrics**.

## 1. Find the results archive, evaluator script, and dataset

Attach the two private Kaggle datasets: Member 2’s 37 MB results ZIP and the small Member 3 final-test kit. Attach the same Galaxy10 file. If the Kaggle upload extracted files, this code handles that as well.

In [ ]:
import os
from pathlib import Path

ROOTS=[Path('/kaggle/input'),Path('/kaggle/working')]
def find_files(pattern):
    return [p for r in ROOTS if r.exists() for p in r.rglob(pattern) if p.is_file()]

print('Member 2 archives:',find_files('member2_full_AB_review.zip')[:5])
print('Evaluation scripts:',find_files('member3_final_test_eval.py')[:5])
print('Galaxy10 H5:',find_files('*.h5')[:12])


## 2. Choose the correct file paths

`DATASET` must be Member 2’s **exact** 2.7 GB HDF5 file. The checksum is verified in the next step. If multiple H5 files appear, change the path deliberately rather than choosing automatically.

In [ ]:
from pathlib import Path
import shutil, zipfile

WORK=Path('/kaggle/working/member3_final_eval_work')
WORK.mkdir(parents=True, exist_ok=True)

scripts=find_files('member3_final_test_eval.py')
if not scripts:
    kit_zips=find_files('member3_final_test_kit.zip')
    assert kit_zips, 'Attach the Member 3 final test kit private Kaggle dataset.'
    with zipfile.ZipFile(kit_zips[0]) as z: z.extractall(WORK)
    scripts=find_files('member3_final_test_eval.py')
assert scripts,'Evaluator script not found.'
SCRIPT=WORK/'member3_final_test_eval.py'
if scripts[0].resolve()!=SCRIPT.resolve():shutil.copyfile(scripts[0], SCRIPT)

zips=find_files('member2_full_AB_review.zip')
if zips:
    ARCHIVE=zips[0]
else:
    files=find_files('member2_run_code_commit.txt')
    assert files,'Attach Member 2 full A/B results ZIP or its extracted directory.'
    ARCHIVE=files[0].parent

h5=find_files('Galaxy10_DECaLS.h5')+find_files('Galaxy10_DECals_NoDuplicated.h5')
assert h5,'Attach the same Galaxy10 HDF5 dataset as Member 2.'
DATASET=h5[0]   # If more than one file was found, inspect and edit this path explicitly
print('SCRIPT:',SCRIPT,'\nARCHIVE:',ARCHIVE,'\nDATASET:',DATASET)
print('HDF5 candidate size:',round(DATASET.stat().st_size/1e9,3),'GB')


## 3. Prepare the exact training code commit (no model updates)

Internet must be enabled for the repository clone. This uses a *separate checkout* and pins the same commit recorded by Member 2.

In [ ]:
import subprocess,sys
REPO=WORK/'Research_Methodology_frozen'
COMMIT='25882a893fe9980acc430008d4869b9a411bb3e0'
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','https://github.com/purveee04/Research_Methodology.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin',COMMIT],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',COMMIT],check=True)
actual=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert actual==COMMIT
print('Pinned training code:',actual)


## 4. PRETEST — integrity and checkpoint-selection verification

This reads the HDF5 hash and split metadata but **does not make any test predictions**. Review the output before authorizing inference.

In [ ]:
import subprocess,sys
OUT=Path('/kaggle/working/member3_final_test_results')
base=[sys.executable,str(SCRIPT),'--repo',str(REPO),'--archive',str(ARCHIVE),'--dataset',str(DATASET),'--output',str(OUT)]
preflight=subprocess.run(base,text=True,capture_output=True)
print(preflight.stdout)
print(preflight.stderr)
assert preflight.returncode==0, 'Preflight failed. STOP: do not perform test inference.'
assert (OUT/'PRETEST_VERIFICATION.json').is_file()
print('PRETEST VERIFIED; no test inference has occurred.')


## 5. ONE-TIME HELD-OUT TEST EVALUATION

**Run this cell only after Step 4 passes and the team confirms the frozen A/B checkpoints and protocol.** It uses both selected `best_clf` classifiers and the **same** 1,731 corrected test images. No training or tuning.

To proceed, change `AUTHORIZE_ONE_FINAL_TEST = False` to `True`. The script will write a `TEST_STARTED.json` marker and refuse repeated test inference with the same output directory.

In [ ]:
AUTHORIZE_ONE_FINAL_TEST=False    # Set True ONCE after reviewing preflight
if AUTHORIZE_ONE_FINAL_TEST:
    assert not (OUT/'FINAL_TEST_RESULTS.json').exists(), 'Final results already exist; do not repeat test evaluation.'
    subprocess.run(base+['--confirm-test-once'],check=True)
else:
    print('NOT RUN: final test remains untouched.')


## 6. View and download final outputs

Only after the one-time test run. Keep validation-best scores separate from test metrics in the research report.

In [ ]:
from IPython.display import display,FileLink
import pandas as pd, shutil
final=OUT/'FINAL_TEST_COMPARISON.csv'
if final.exists():
    display(pd.read_csv(final))
    display(pd.read_csv(OUT/'FINAL_TEST_PER_CLASS.csv'))
    resultzip=shutil.make_archive('/kaggle/working/member3_final_test_handoff','zip',str(OUT))
    print('Result ZIP:',resultzip)
    display(FileLink(resultzip))
else:
    print('No final test predictions yet. Run Step 5 only after verification.')
